In [1]:
import os
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer

In [15]:
directory = 'output_files/output_v3_16_05'

unique_classifications = set()

for filename in os.listdir(directory):
    if filename.endswith('.csv'):
        filepath = os.path.join(directory, filename)
        try:
            df = pd.read_csv(filepath)
            if 'Classification' in df.columns:
                for val in df['Classification'].dropna().unique():
                    cleaned_val = str(val).replace('*', '').replace(' ', '')
                    unique_classifications.add(cleaned_val)
            else:
                print(f"'Classification' column not found in {filename}")
        except Exception as e:
            print(f"Error processing {filename}: {e}")

print("Unique values in 'Classification' column:")
for value in sorted(unique_classifications):
    print(value)

Unique values in 'Classification' column:

'CR
()-newclassification
(CR)RR
(NOROADACCESS)
-
-2ndclass
-3rdclass
.
15.0
17.0
17VicinityextendedtoIbobdry/cemetry
18VicinityextendedtoIbobdry/cemetery
19Correctedvicinity(OsmenaSt.included
20Correctedvicinity(combined)
21Correctedvicinity(OsmenaSt.includedin
22Newlyaddedstreet(extension)
23Newclassification
2ndClass
2ndclass
37
3rdClass
3rdclass
50
=
A
A-36B
A0
A1
A1-1
A1-1stclass
A1-2
A1-2ndclass
A1-3
A1-4
A10
A10-QUARRY
A10/A15
A101
A102
A103
A11
A12
A121
A1210
A1211
A1212
A1213
A1214
A1215
A1216
A1217
A1218
A1219
A122
A123
A124
A125
A126
A127
A128
A129
A13
A13-1
A13-2
A13-3
A14
A14-1
A14-2
A14-3
A14-4
A141
A142
A143
A144
A15
A15-1
A15/A10
A16
A16-1
A16-2
A16-3
A16-4
A17
A17-1
A17-2
A17-3
A18
A18-1
A18-2
A18-3
A19
A19-1
A19-2
A19-3
A19-4
A2
A2-1
A2-2
A2-3
A2-4
A20
A20-1
A20-2
A20-3
A201
A202
A21
A21-1
A21-2
A21-3
A22
A22-1
A22-2
A22-3
A23
A23-1
A23-2
A23-3
A231
A232
A24
A25
A25-1
A25-2
A25-3
A25-4
A251
A252
A26
A261
A262
A263
A27
A28
A29


In [ ]:
BASE_CODES = {
    # original legend
    'RR', 'CR', 'RC', 'CC', 'CL',
    'A',                           # agricultural root
    'GL', 'GP', 'I', 'X',
    'APD', 'PS', 'DA',
    'C', 'AI', 'AR', 'IR',
}

FREE_TOKENS = {
    'AGRI', 'CEMETERY', 'MEMORIALLOT', 'MINERALLAND',
}

In [ ]:
AGRI_CODE_RE   = r'A\d{1,4}'

CODE_TOKEN_RE  = rf'(?:{AGRI_CODE_RE}|' + '|'.join(sorted(BASE_CODES)) + r')'

In [ ]:
# Master Patterns
VALID_PATTERNS = [
    rf'^{CODE_TOKEN_RE}$',                           # plain code
    rf'^{CODE_TOKEN_RE}\.$',                         # code + dot        (APD.)
    rf'^{CODE_TOKEN_RE}\d+[A-Z]*$',                  # code + digits     (RR23, CR1, I2, A5023)
    rf'^{CODE_TOKEN_RE}[.-]\d+[A-Z]*$',              # code-sep-digits   (RR-1, A3.1)
    rf'^{CODE_TOKEN_RE}[./-][A-Z]+[A-Z0-9]*(?:/[A-Z]+[A-Z0-9]*)*$',   # code-desc(/desc)*  (A50-PIGGERY/POULTRY)
    rf'^{CODE_TOKEN_RE}/\d{{1,4}}$',                 # A49/50
    rf'^{CODE_TOKEN_RE}/' + CODE_TOKEN_RE + r'$',    # A10/A15  or APD-CR
    rf'^{CODE_TOKEN_RE}\([^)]+\)$',                  # code(text)        (RR(Inner))
    rf'^{AGRI_CODE_RE}[A-Z]+$',                      # A5O, A3C
    rf'^(?:' + '|'.join(sorted(FREE_TOKENS)) + r')$',# AGRI, CEMETERY …
]

VALID_RE = re.compile('|'.join(f'(?:{pat})' for pat in VALID_PATTERNS), re.I)

In [ ]:
def is_valid_classification(raw: str) -> bool:
    if not raw:
        return False
    token = raw.strip().lstrip("'\"").upper()
    return bool(VALID_RE.fullmatch(token))

In [16]:
filtered_classifications = {
    val for val in unique_classifications if is_valid_classification(val)
}

In [17]:
unique_classifications - filtered_classifications

{'',
 '()-newclassification',
 '(CR)RR',
 '(NOROADACCESS)',
 '-',
 '-2ndclass',
 '-3rdclass',
 '.',
 '15.0',
 '17.0',
 '17VicinityextendedtoIbobdry/cemetry',
 '18VicinityextendedtoIbobdry/cemetery',
 '19Correctedvicinity(OsmenaSt.included',
 '20Correctedvicinity(combined)',
 '21Correctedvicinity(OsmenaSt.includedin',
 '22Newlyaddedstreet(extension)',
 '23Newclassification',
 '2ndClass',
 '2ndclass',
 '37',
 '3rdClass',
 '3rdclass',
 '50',
 '=',
 '============',
 '=============',
 'CLASS',
 'CLASSI-',
 'CLASSI-FICATION',
 'CLASSIFI-CATION',
 'CLASSIFICATION',
 'D.O.NO.',
 'D.O.No.',
 'DONO.',
 'EXISTING/APPROVEDZONALVALUE\r\n(D.O.No.30-18)\r\n(July23,2018)',
 'EffectivityDate',
 'FICATION',
 'GERONIMO',
 'MASCAP(MOUNTAIN)',
 'NOTEXISTING',
 'PC',
 'PH',
 'PLEASESEEIDENTIFIEDSTREETS',
 'R',
 'R1',
 'RIZAL',
 'RODRIGUEZ(MONTALBAN)',
 'ROSARIO',
 'SANJOSE',
 'SANRAFAEL',
 'WC',
 '^PerOcularInspetionthelocationisPredominantlyCommercial',
 'includedinvicinity)',
 'vicinity)',
 'Í',
 'ßß',
 '

In [18]:
filtered_classifications

{"'CR",
 'A',
 'A-36B',
 'A0',
 'A1',
 'A1-1',
 'A1-1stclass',
 'A1-2',
 'A1-2ndclass',
 'A1-3',
 'A1-4',
 'A10',
 'A10-QUARRY',
 'A10/A15',
 'A101',
 'A102',
 'A103',
 'A11',
 'A12',
 'A121',
 'A1210',
 'A1211',
 'A1212',
 'A1213',
 'A1214',
 'A1215',
 'A1216',
 'A1217',
 'A1218',
 'A1219',
 'A122',
 'A123',
 'A124',
 'A125',
 'A126',
 'A127',
 'A128',
 'A129',
 'A13',
 'A13-1',
 'A13-2',
 'A13-3',
 'A14',
 'A14-1',
 'A14-2',
 'A14-3',
 'A14-4',
 'A141',
 'A142',
 'A143',
 'A144',
 'A15',
 'A15-1',
 'A15/A10',
 'A16',
 'A16-1',
 'A16-2',
 'A16-3',
 'A16-4',
 'A17',
 'A17-1',
 'A17-2',
 'A17-3',
 'A18',
 'A18-1',
 'A18-2',
 'A18-3',
 'A19',
 'A19-1',
 'A19-2',
 'A19-3',
 'A19-4',
 'A2',
 'A2-1',
 'A2-2',
 'A2-3',
 'A2-4',
 'A20',
 'A20-1',
 'A20-2',
 'A20-3',
 'A201',
 'A202',
 'A21',
 'A21-1',
 'A21-2',
 'A21-3',
 'A22',
 'A22-1',
 'A22-2',
 'A22-3',
 'A23',
 'A23-1',
 'A23-2',
 'A23-3',
 'A231',
 'A232',
 'A24',
 'A25',
 'A25-1',
 'A25-2',
 'A25-3',
 'A25-4',
 'A251',
 'A252',
 'A26'

In [24]:
CSV_DIR = 'output_files/output_v3_16_05'
ENCODING = 'utf-8'

In [25]:
street_records = []
vicinity_records = []

for fname in os.listdir(CSV_DIR):
    if not fname.lower().endswith('.csv'):
        continue
    path = os.path.join(CSV_DIR, fname)
    try:
        df = pd.read_csv(path, encoding=ENCODING)
    except UnicodeDecodeError:
        df = pd.read_csv(path, encoding='latin-1')

    for col in ['Street/Subdivision', 'Vicinity']:
        if col not in df.columns:
            df[col] = ''

    street_clean = df['Street/Subdivision'].fillna('').astype(str).str.strip().str.replace(r'\s+', ' ', regex=True)
    vicinity_clean = df['Vicinity'].fillna('').astype(str).str.strip().str.replace(r'\s+', ' ', regex=True)

    street_records.extend(street_clean.tolist())
    vicinity_records.extend(vicinity_clean.tolist())

print(f'Total records: {len(street_records):,}')

Total records: 831,842


In [30]:
# TF–IDF PER COLUMN
def compute_tfidf(records, label):
    vectorizer = TfidfVectorizer(lowercase=True)
    tfidf_matrix = vectorizer.fit_transform(records)
    vocab = vectorizer.get_feature_names_out()
    print(f'\n[{label}] Vocabulary size: {len(vocab):,}')
    print(f'[{label}] TFIDF matrix shape: {tfidf_matrix.shape}')
    
    # Top global terms by mean TF-IDF weight
    mean_tfidf = tfidf_matrix.mean(axis=0).A1
    top_idx = mean_tfidf.argsort()[::-1][:50]
    print(f'\nTop 50 terms in [{label}]:')
    for rank, idx in enumerate(top_idx, 1):
        print(f'{rank:2d}. {vocab[idx]:20s}  {mean_tfidf[idx]:.4f}')
    
    # Inspect a specific record
    doc_id = 0
    row = tfidf_matrix[doc_id]
    nonzero = row.nonzero()[1]
    terms_weights = sorted(
        ((vocab[i], row[0, i]) for i in nonzero),
        key=lambda x: x[1],
        reverse=True
    )[:20]

    print(f'\nTop terms for record {doc_id} in [{label}]:')
    for term, wt in terms_weights:
        print(f'  {term:20s} {wt:.4f}')

In [31]:
compute_tfidf(street_records, 'Street/Subdivision')


[Street/Subdivision] Vocabulary size: 23,881
[Street/Subdivision] TFIDF matrix shape: (831842, 23881)

Top 50 terms in [Street/Subdivision]:
 1. all                   0.2970
 2. lots                  0.2953
 3. other                 0.0891
 4. road                  0.0872
 5. streets               0.0795
 6. barangay              0.0589
 7. provincial            0.0353
 8. national              0.0302
 9. agricultural          0.0273
10. st                    0.0264
11. along                 0.0218
12. roads                 0.0193
13. municipal             0.0184
14. highway               0.0152
15. lot                   0.0134
16. street                0.0076
17. subdivision           0.0053
18. interior              0.0049
19. formerly              0.0047
20. subd                  0.0044
21. san                   0.0039
22. rizal                 0.0035
23. condominium           0.0026
24. ave                   0.0026
25. ipil                  0.0022
26. avenue                0.0021


In [32]:
compute_tfidf(vicinity_records, 'Vicinity')


[Vicinity] Vocabulary size: 16,772
[Vicinity] TFIDF matrix shape: (831842, 16772)

Top 50 terms in [Vicinity]:
 1. interior              0.2376
 2. lots                  0.0999
 3. road                  0.0809
 4. along                 0.0796
 5. barangay              0.0464
 6. the                   0.0277
 7. provincial            0.0224
 8. national              0.0185
 9. all                   0.0182
10. st                    0.0176
11. other                 0.0170
12. streets               0.0146
13. municipal             0.0122
14. lot                   0.0117
15. land                  0.0106
16. highway               0.0093
17. roads                 0.0091
18. brgy                  0.0085
19. access                0.0083
20. without               0.0082
21. to                    0.0075
22. local                 0.0046
23. within                0.0039
24. of                    0.0037
25. ave                   0.0035
26. cassava               0.0033
27. etc                   0.00